# Student Performance Dataset Analysis

This notebook performs the initial inspection of the uploaded dataset. It does not modify the data, preprocess features, or train a model.

## 1. Import Libraries

Import pandas and numpy for data inspection, plus matplotlib and seaborn for basic visual exploration.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 2. Load Dataset

The CSV is loaded directly from the raw data directory. No values or columns are changed.

In [ ]:
data_path = "../data/raw/StudentPerformanceFactors.csv"
df = pd.read_csv(data_path)

df.head()

## 3. Basic Dataset Information

This section checks the dataset dimensions, sample records, column names, and pandas data types.

In [ ]:
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")

print("First 5 rows:")
display(df.head())

print("Last 5 rows:")
display(df.tail())

print("Column names:")
print(df.columns.tolist())

print("DataFrame information:")
df.info()

## 4. Numerical Features

Numerical columns are identified from the dataframe dtypes. Descriptive statistics are shown without altering the data.

In [ ]:
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("Descriptive statistics for numerical columns:")
display(df[numerical_columns].describe())

## 5. Categorical Features

Categorical columns are identified from the dataframe dtypes. Their unique values and cardinalities are displayed for initial inspection.

In [ ]:
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

categorical_summary = pd.DataFrame({
    "unique_count": df[categorical_columns].nunique(dropna=False),
    "unique_values": [df[column].unique().tolist() for column in categorical_columns],
})

display(categorical_summary)

## 6. Missing Values

The following table reports missing values in each column. No missing-value treatment is performed.

In [ ]:
missing_values = df.isna().sum().to_frame(name="missing_count")
missing_values["missing_percentage"] = (missing_values["missing_count"] / len(df) * 100).round(2)

display(missing_values)

## 7. Duplicate Records

Duplicate rows are counted without removing or changing them.

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

## 8. Target Variable Analysis

The dataset contains `Exam_Score`, which is treated as the target column for this initial analysis because it is the outcome column present in the uploaded data.

In [ ]:
target_column = "Exam_Score"
target = df[target_column]

print(f"Target column: {target_column}")
print("Target distribution:")
display(target.value_counts().sort_index().to_frame(name="count"))

print("Target descriptive statistics:")
display(target.describe())

print("Target summary:")
target_summary = pd.Series({
    "minimum": target.min(),
    "maximum": target.max(),
    "mean": target.mean(),
    "median": target.median(),
    "standard_deviation": target.std(),
})
display(target_summary.to_frame(name=target_column))

## 9. Initial Findings

- The dataset contains 6,607 rows and 20 columns.
- There are 7 numerical columns and 13 categorical columns.
- There are 235 missing values in total, found in `Teacher_Quality`, `Parental_Education_Level`, and `Distance_from_Home`.
- There are no duplicate rows.
- `Exam_Score` is the observed target column in this dataset.
- `Exam_Score` ranges from 55 to 101, with a mean of 67.24, a median of 67.0, and a standard deviation of 3.89.
- No encoding, feature engineering, normalization, train-test split, or model training has been performed.

This notebook intentionally stops after initial dataset analysis. Preprocessing and later machine learning stages are outside its scope.

## 10. Preprocessing Plan

### Target variable

`Exam_Score` is the target variable. It remains numeric and is not converted into categories such as High, Average, or Low. It is excluded from the predictor matrix `X`; the target vector is `y`.

### Predictor feature types

The predictor columns are identified automatically from the dataframe after removing `Exam_Score`.

- Numerical predictors: `Hours_Studied`, `Attendance`, `Sleep_Hours`, `Previous_Scores`, `Tutoring_Sessions`, and `Physical_Activity`
- Categorical predictors: `Parental_Involvement`, `Access_to_Resources`, `Extracurricular_Activities`, `Motivation_Level`, `Internet_Access`, `Family_Income`, `Teacher_Quality`, `School_Type`, `Peer_Influence`, `Learning_Disabilities`, `Parental_Education_Level`, `Distance_from_Home`, and `Gender`

### Missing-value strategy

The categorical columns with missing values (`Teacher_Quality`, `Parental_Education_Level`, and `Distance_from_Home`) use the most frequent category. This is reproducible, preserves an existing category, and avoids inventing a new label. Numerical predictors use median imputation only if missing numerical values are encountered. The target is never imputed.

### Encoding strategy

Categorical predictors will be imputed and one-hot encoded with an sklearn `Pipeline` inside a `ColumnTransformer`. `handle_unknown="ignore"` allows later validation or future data to contain categories that were not present during training.

### Scaling strategy

Numerical predictors will be median-imputed and standardized with `StandardScaler`. Scaling supports later scale-sensitive model choices while keeping this preprocessing reusable; it does not determine the final algorithm.

### Leakage prevention

The preprocessing transformer is created but not fitted in `src/preprocessing.py`. After a later train-test split, it must be fitted only on `X_train` and then used to transform validation or test data. The saved `data/processed/cleaned_data.csv` is human-readable and contains imputed original columns, not an encoded or scaled matrix.